In [1]:
import pandas as pd
import numpy as np
import cv2
import os
import tqdm
import h5py
from sklearn.model_selection import train_test_split

In [2]:
mat_folder = "./brain_tumor/Dataset/brainTumorSegmentationData"
output_folder = "./brain_tumor/Dataset/segmentation_data"
csv_output_path = "./brain_tumor/Dataset/brain_tumor_metadata.csv"

In [3]:
os.makedirs(output_folder,exist_ok=True)

In [4]:
label_map = {
    1:"meningioma",
    2:"glioma",
    3:"pituitary_tumor"
}

In [5]:
data = []

files = [f for f in os.listdir(mat_folder) if f.endswith(".mat")]

In [6]:
for file in tqdm.tqdm(files):
    file_path = os.path.join(mat_folder,file)
    
    with h5py.File(file_path,'r') as f:
        cj = f['cjdata']
        
        label = int(cj['label'][0][0])
        pid = ''.join(chr(int(x)) for x in np.array(cj['PID']).flatten())
        
        data.append({
            "file":file,
            "label":label_map[label],
            "patient_id":pid,
            "path":file_path
        })

df = pd.DataFrame(data)

print("Total Samples:",len(df))
df.head()

  0%|          | 0/3064 [00:00<?, ?it/s]

100%|██████████| 3064/3064 [00:46<00:00, 65.29it/s]

Total Samples: 3064


,file,label,patient_id,path
0,1.mat,meningioma,100360,./brain_tumor/Dataset/brainTumorSegmentationDa...
1,10.mat,meningioma,101016,./brain_tumor/Dataset/brainTumorSegmentationDa...
2,100.mat,meningioma,107494,./brain_tumor/Dataset/brainTumorSegmentationDa...
3,1000.mat,pituitary_tumor,112649,./brain_tumor/Dataset/brainTumorSegmentationDa...
4,1001.mat,pituitary_tumor,112649,./brain_tumor/Dataset/brainTumorSegmentationDa...


In [7]:
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)

val_df, test_df = train_test_split(temp_df, test_size=0.66, stratify=temp_df['label'], random_state=42)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

final_df = pd.concat([train_df,val_df,test_df])

print(final_df['split'].value_counts())

split
train    2144
test      608
val       312
Name: count, dtype: int64


In [8]:
image_paths = []
mask_paths = []

index = 0

for _,row in tqdm.tqdm(final_df.iterrows(),total = len(final_df)):
    
    index += 1
    file_path = row['path']
    split = row['split']
    label = row['label']
    
    with h5py.File(file_path,'r') as f:
        image = np.array(f['cjdata']['image'])
        mask = np.array(f['cjdata']['tumorMask'])
    
    image = image - image.min()
    image = image / (image.max() + 1e-8)
    image = (image * 255).astype(np.uint8)
    
    mask = mask - mask.min()
    mask = mask / (mask.max() + 1e-8)
    mask = (mask * 255).astype(np.uint8)
    
    img_save_dir = os.path.join(output_folder,split,'image',label)
    mask_save_dir = os.path.join(output_folder,split,'mask',label)
    
    os.makedirs(img_save_dir,exist_ok=True)
    os.makedirs(mask_save_dir,exist_ok=True)
    
    image_name = str(index) + ".png"
    mask_name = str(index) + "_mask.png"
    
    img_save_path = os.path.join(img_save_dir,image_name)
    mask_save_path = os.path.join(mask_save_dir,mask_name)
    
    image_paths.append(img_save_path)
    mask_paths.append(mask_save_path)
    
    cv2.imwrite(img_save_path,image)
    cv2.imwrite(mask_save_path,mask)

100%|██████████| 3064/3064 [01:24<00:00, 36.40it/s]


In [9]:
final_df = final_df.reset_index(drop=True)
final_df["image_path"] = image_paths
final_df["mask_path"] = mask_paths

final_df.to_csv(csv_output_path, index=False)

In [10]:
final_df.head()

,file,label,patient_id,path,split,image_path,mask_path
0,2983.mat,glioma,MR051644C,./brain_tumor/Dataset/brainTumorSegmentationDa...,train,./brain_tumor/Dataset/segmentation_data\train\...,./brain_tumor/Dataset/segmentation_data\train\...
1,2441.mat,glioma,MR048994,./brain_tumor/Dataset/brainTumorSegmentationDa...,train,./brain_tumor/Dataset/segmentation_data\train\...,./brain_tumor/Dataset/segmentation_data\train\...
2,858.mat,glioma,108479,./brain_tumor/Dataset/brainTumorSegmentationDa...,train,./brain_tumor/Dataset/segmentation_data\train\...,./brain_tumor/Dataset/segmentation_data\train\...
3,752.mat,glioma,100820,./brain_tumor/Dataset/brainTumorSegmentationDa...,train,./brain_tumor/Dataset/segmentation_data\train\...,./brain_tumor/Dataset/segmentation_data\train\...
4,235.mat,meningioma,101797,./brain_tumor/Dataset/brainTumorSegmentationDa...,train,./brain_tumor/Dataset/segmentation_data\train\...,./brain_tumor/Dataset/segmentation_data\train\...


In [11]:
final_df['image_path'].duplicated().sum()

0